# 📈 Customer Lifetime Value (CLV) Prediction System

## Comprehensive Analysis: From Probabilistic Models to ML Predictions

---

**Author**: CLV Analytics Team  
**Date**: 2024  
**Version**: 1.0

## 1. Introduction: What is CLV and Why It Matters

### 🎯 Learning Objectives

By the end of this notebook, you will understand:

1. **What Customer Lifetime Value (CLV) means** and why it's critical for business
2. **The 80/20 rule** - how a small percentage of customers drive most revenue
3. **RFM Analysis** - a powerful segmentation technique
4. **BG/NBD + Gamma-Gamma models** - probabilistic CLV modeling
5. **ML-based CLV prediction** - using XGBoost with feature engineering
6. **How to apply these insights** for marketing and retention strategies

### 📊 The 80/20 Rule in Customer Value

Also known as the **Pareto Principle**, the 80/20 rule suggests that:

- **80% of revenue** comes from **20% of customers**
- **20% of revenue** comes from **80% of customers**

This means identifying and retaining your high-value customers is **critical** for business success.

### 💰 Why CLV Matters

**Customer Acquisition Cost (CAC)** vs **CLV**:

| Metric | Definition | Why It Matters |
|--------|------------|----------------|
| CAC | Cost to acquire a new customer | Should be < CLV for profitability |
| CLV | Total revenue from a customer over time | Determines marketing budget |
| CLV:CAC | Ratio of CLV to acquisition cost | Target: 3:1 or higher |

**CLV Applications:**

1. **Marketing Budget Allocation** - Focus on high-CLV segments
2. **Customer Acquisition ROI** - Know how much to spend per customer
3. **Retention Program Targeting** - Prioritize at-risk high-value customers
4. **Product Development** - Understand customer preferences by segment
5. **Personalization** - Tailor offers based on CLV potential

In [ ]:
# Install required packages
!pip install -q lifetimes xgboost lightgbm optuna shap plotly pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 2. Loading and Exploring the Data

We'll use the **Online Retail II dataset** from UCI:

- **1M+ transactions**
- **Multiple countries**
- **Date + monetary information** - perfect for CLV analysis

In [ ]:
# Load data
df = pd.read_csv('data/online_retail.csv', encoding='latin-1')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
# Basic stats
df.head(10)

In [ ]:
# Date range
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"\nUnique customers: {df['Customer ID'].nunique()}")
print(f"Total transactions: {len(df):,}")

## 3. Exploratory Data Analysis (EDA)

Let's understand the distribution of key metrics:

In [ ]:
# Revenue distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Revenue per transaction
df['Revenue'] = df['Quantity'] * df['Price']
ax1 = axes[0, 0]
df['Revenue'].hist(bins=50, ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('Revenue Distribution (All Transactions)', fontsize=12)
ax1.set_xlabel('Revenue (£)')
ax1.set_ylabel('Count')
ax1.set_xlim(0, df['Revenue'].quantile(0.99))

# 2. Log-scale revenue
ax2 = axes[0, 1]
np.log1p(df['Revenue']).hist(bins=50, ax=ax2, color='coral', edgecolor='white')
ax2.set_title('Log-Revenue Distribution', fontsize=12)
ax2.set_xlabel('Log(Revenue + 1)')
ax2.set_ylabel('Count')

# 3. Customer lifetime distribution
customer_lifetime = df.groupby('Customer ID')['InvoiceDate'].agg(['min', 'max'])
customer_lifetime['lifetime_days'] = (customer_lifetime['max'] - customer_lifetime['min']).dt.days
ax3 = axes[1, 0]
customer_lifetime['lifetime_days'].hist(bins=50, ax=ax3, color='seagreen', edgecolor='white')
ax3.set_title('Customer Lifetime Distribution', fontsize=12)
ax3.set_xlabel('Days between first and last purchase')
ax3.set_ylabel('Count')

# 4. Purchase frequency
purchase_counts = df.groupby('Customer ID')['Invoice'].nunique()
ax4 = axes[1, 1]
purchase_counts.hist(bins=50, ax=ax4, color='purple', edgecolor='white')
ax4.set_title('Purchase Frequency Distribution', fontsize=12)
ax4.set_xlabel('Number of Transactions')
ax4.set_ylabel('Count')
ax4.set_xlim(0, purchase_counts.quantile(0.99))

plt.tight_layout()
plt.show()

### Key Insights from EDA:

1. **Revenue is highly right-skewed** - Most transactions are small, few are large
2. **Customer lifetime varies widely** - From days to years
3. **Most customers make few purchases** - Long-tail distribution

## 4. Data Preprocessing

Cleaning the data is crucial for accurate CLV modeling.

In [ ]:
def preprocess_data(df):
    """Clean and preprocess the retail data."""
    df = df.copy()
    
    # Remove cancelled orders (Invoice starts with 'C')
    df = df[~df['Invoice'].astype(str).str.startswith('C')]
    
    # Remove negative quantities and prices
    df = df[df['Quantity'] > 0]
    df = df[df['Price'] > 0]
    
    # Remove rows with missing Customer ID
    df = df[df['Customer ID'].notna()]
    df['Customer ID'] = df['Customer ID'].astype(int)
    
    # Calculate revenue
    df['Revenue'] = df['Quantity'] * df['Price']
    
    # Remove extreme outliers
    price_99th = df['Price'].quantile(0.99)
    df = df[df['Price'] <= price_99th]
    
    print(f"Cleaned data: {len(df):,} transactions from {df['Customer ID'].nunique():,} customers")
    
    return df

df_clean = preprocess_data(df)
print(f"\nCleaned dataset shape: {df_clean.shape}")

In [ ]:
# Create train/test split (observation period vs holdout)
def create_train_test_split(df, observation_days=365, holdout_days=90):
    max_date = df['InvoiceDate'].max()
    observation_end = max_date - pd.Timedelta(days=holdout_days)
    holdout_start = observation_end + pd.Timedelta(days=1)
    
    train_df = df[df['InvoiceDate'] <= observation_end]
    test_df = df[df['InvoiceDate'] >= holdout_start]
    
    print(f"Observation period: {df['InvoiceDate'].min()} to {observation_end}")
    print(f"Holdout period: {holdout_start} to {max_date}")
    print(f"Train transactions: {len(train_df):,}")
    print(f"Test transactions: {len(test_df):,}")
    
    return train_df, test_df

train_df, test_df = create_train_test_split(df_clean)

## 5. RFM Analysis

RFM stands for **Recency, Frequency, Monetary** - the three pillars of customer segmentation.

| Component | Definition | Why It Matters |
|-----------|------------|----------------|
| **Recency** | Days since last purchase | Recent buyers are more likely to buy again |
| **Frequency** | Number of transactions | Frequent buyers are more engaged |
| **Monetary** | Average transaction value | Higher spend = higher value |

In [ ]:
def compute_rfm(df, snapshot_date=None):
    """Compute RFM metrics for each customer."""
    if snapshot_date is None:
        snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
    
    rfm = df.groupby('Customer ID').agg({
        'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
        'Invoice': 'nunique',  # Frequency
        'Revenue': ['sum', 'mean']  # Monetary
    })
    
    rfm.columns = ['Recency', 'Frequency', 'TotalRevenue', 'AvgTransactionValue']
    rfm = rfm.reset_index()
    
    return rfm

rfm = compute_rfm(train_df)
print(f"RFM computed for {len(rfm):,} customers")
rfm.head()

In [ ]:
# Score RFM (1-5 scale)
def score_rfm(rfm_df):
    """Score RFM components on 1-5 scale."""
    rfm_df = rfm_df.copy()
    
    # Recency: lower is better
    rfm_df['R_score'] = pd.qcut(rfm_df['Recency'].rank(method='first'), 
                                q=5, labels=[5, 4, 3, 2, 1]).astype(int)
    
    # Frequency: higher is better
    rfm_df['F_score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), 
                                q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    
    # Monetary: higher is better
    rfm_df['M_score'] = pd.qcut(rfm_df['AvgTransactionValue'].rank(method='first'), 
                                q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    
    # Combined score
    rfm_df['RFM_score'] = rfm_df['R_score'] * 100 + rfm_df['F_score'] * 10 + rfm_df['M_score']
    
    return rfm_df

rfm_scored = score_rfm(rfm)
rfm_scored.head(10)

In [ ]:
# Segment customers
def segment_customers(rfm_df):
    """Assign RFM segment to each customer."""
    def assign_segment(row):
        r, f, m = row['R_score'], row['F_score'], row['M_score']
        
        if r >= 4 and f >= 4 and m >= 4:
            return 'Champions'
        elif f >= 4:
            return 'Loyal'
        elif r >= 3 and f >= 2:
            return 'Potential Loyalists'
        elif r >= 4 and f <= 1:
            return 'New Customers'
        elif r <= 2 and f >= 3:
            return 'At Risk'
        elif r <= 2 and f <= 2:
            return 'Hibernating'
        elif r == 1 and f == 1:
            return 'Lost'
        else:
            return 'Needs Attention'
    
    rfm_df = rfm_df.copy()
    rfm_df['segment'] = rfm_df.apply(assign_segment, axis=1)
    
    return rfm_df

rfm_segmented = segment_customers(rfm_scored)

# Visualize segments
segment_counts = rfm_segmented['segment'].value_counts()

fig = px.bar(
    x=segment_counts.index,
    y=segment_counts.values,
    color=segment_counts.index,
    title='Customer Distribution by RFM Segment'
)
fig.update_layout(xaxis_title='Segment', yaxis_title='Number of Customers', showlegend=False)
fig.show()

In [ ]:
# Segment statistics
segment_stats = rfm_segmented.groupby('segment').agg({
    'Customer ID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'AvgTransactionValue': 'mean',
    'TotalRevenue': ['sum', 'mean']
}).round(2)

segment_stats.columns = ['Count', 'Avg_Recency', 'Avg_Frequency', 
                         'Avg_Monetary', 'Total_Revenue', 'Avg_Revenue']

segment_stats['Revenue_Share_%'] = (segment_stats['Total_Revenue'] / segment_stats['Total_Revenue'].sum() * 100).round(2)

segment_stats = segment_stats.sort_values('Total_Revenue', ascending=False)
segment_stats

In [ ]:
# RFM Heatmap: Recency vs Frequency with Monetary values
pivot = rfm_segmented.groupby(['R_score', 'F_score'])['AvgTransactionValue'].mean().unstack()

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=[f'F={i}' for i in pivot.columns],
    y=[f'R={i}' for i in pivot.index],
    colorscale='RdYlGn',
    colorbar_title='Avg Monetary (£)'
))

fig.update_layout(
    title='RFM Heatmap: Recency vs Frequency<br><sub>Cell values show average transaction value</sub>',
    xaxis_title='Frequency Score',
    yaxis_title='Recency Score'
)

fig.show()

## 6. Cohort Analysis

Cohort analysis tracks customer behavior over time based on their first purchase month.

In [ ]:
def create_cohort_analysis(df):
    """Create cohort retention analysis."""
    df = df.copy()
    
    # Get first purchase date per customer
    first_purchase = df.groupby('Customer ID')['InvoiceDate'].min()
    df['CohortMonth'] = df['Customer ID'].map(first_purchase).dt.to_period('M')
    
    # Get transaction month
    df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
    
    # Calculate cohort index (months since first purchase)
    df['CohortIndex'] = (df['InvoiceMonth'].astype('int64') - df['CohortMonth'].astype('int64'))
    
    # Count unique customers per cohort and month
    cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
    cohort_data.columns = ['CohortMonth', 'CohortIndex', 'n_customers']
    
    # Pivot to create retention matrix
    cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='n_customers')
    
    # Get cohort sizes
    cohort_sizes = cohort_pivot[0]
    
    # Calculate retention rate
    retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100
    
    return retention_matrix

retention = create_cohort_analysis(train_df)
print(f"Retention matrix shape: {retention.shape}")

In [ ]:
# Visualize retention matrix
fig = go.Figure(data=go.Heatmap(
    z=retention.values[:12, :12],  # First 12 cohorts, 12 months
    x=[f'Month {i}' for i in range(12)],
    y=[str(idx) for idx in retention.index[:12]],
    colorscale='RdYlGn',
    colorbar_title='Retention %',
    text=np.round(retention.values[:12, :12], 1),
    texttemplate='%{text:.1f}%'
))

fig.update_layout(
    title='Monthly Cohort Retention Analysis<br><sub>First 12 cohorts, first 12 months</sub>',
    xaxis_title='Months Since First Purchase',
    yaxis_title='Cohort Month'
)

fig.show()

## 7. Probabilistic CLV Modeling: BG/NBD + Gamma-Gamma

### Understanding the Models

#### **BG/NBD (Beta-Geometric Negative Binomial Distribution)**

The BG/NBD model predicts **how many purchases** a customer will make.

**Key Assumptions:**

1. While active, customer's transactions follow a **Poisson process** (random, independent)
2. After any transaction, customer has probability **p of dropping out**
3. Dropout time follows a **geometric distribution**

**What it tells us:**
- Expected number of purchases in next T days
- Probability a customer is still "alive" (active)

In [ ]:
from lifetimes import BetaGeoFitter, GammaGammaFitter

# Prepare customer features for BG/NBD
def prepare_bgnbd_features(df, snapshot_date=None):
    """Prepare features for BG/NBD model."""
    if snapshot_date is None:
        snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
    
    customer_features = []
    
    for customer_id, group in df.groupby('Customer ID'):
        n_purchases = group['Invoice'].nunique()
        first_date = group['InvoiceDate'].min()
        last_date = group['InvoiceDate'].max()
        
        frequency = n_purchases - 1  # Exclude first purchase
        recency = (last_date - first_date).days
        T = (snapshot_date - first_date).days
        
        customer_features.append({
            'Customer ID': customer_id,
            'frequency': frequency,
            'recency': recency,
            'T': T
        })
    
    return pd.DataFrame(customer_features)

# Prepare features
snapshot_date = train_df['InvoiceDate'].max() + pd.Timedelta(days=1)
bgnbd_features = prepare_bgnbd_features(train_df, snapshot_date)

print(f"Prepared BG/NBD features for {len(bgnbd_features):,} customers")
bgnbd_features.head()

In [ ]:
# Fit BG/NBD model
bgnbd_model = BetaGeoFitter(penalizer_coef=0.01)
bgnbd_model.fit(
    bgnbd_features['frequency'],
    bgnbd_features['recency'],
    bgnbd_features['T']
)

print("BG/NBD Model Parameters:")
print(bgnbd_model.params_)
print(f"\nModel Summary:\n{bgnbd_model.summary}")

In [ ]:
# Predict purchases for next 90 days
bgnbd_features['predicted_purchases_90d'] = bgnbd_model.conditional_expected_number_of_purchases_up_to_time(
    t=90,
    frequency=bgnbd_features['frequency'],
    recency=bgnbd_features['recency'],
    T=bgnbd_features['T']
)

# Predict probability alive
bgnbd_features['prob_alive'] = bgnbd_model.conditional_probability_alive(
    frequency=bgnbd_features['frequency'],
    recency=bgnbd_features['recency'],
    T=bgnbd_features['T']
)

print("Predicted purchases and probability alive:")
bgnbd_features.head(10)

In [ ]:
# Visualize frequency vs recency probability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Probability alive heatmap
frequencies = np.arange(0, 30, 1)
recencies = np.arange(0, 365, 10)

Z = np.zeros((len(recencies), len(frequencies)))
for i, rec in enumerate(recencies):
    for j, freq in enumerate(frequencies):
        if freq > 0:
            Z[i, j] = bgnbd_model.conditional_probability_alive(freq, rec, rec + 1)

ax1 = axes[0]
im = ax1.imshow(Z, cmap='viridis', aspect='auto', 
                extent=[0, 30, 365, 0], origin='lower')
ax1.set_xlabel('Frequency')
ax1.set_ylabel('Recency (days)')
ax1.set_title('Probability Customer is Alive')
plt.colorbar(im, ax=ax1)

# Plot 2: Predicted purchases distribution
ax2 = axes[1]
bgnbd_features['predicted_purchases_90d'].hist(bins=50, ax=ax2, color='steelblue')
ax2.set_xlabel('Predicted Purchases (90 days)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Predicted Purchases')

plt.tight_layout()
plt.show()

### **Gamma-Gamma Model**

The Gamma-Gamma model predicts **average transaction value**.

**Key Assumptions:**

1. Transaction values follow a **Gamma distribution**
2. The shape parameter varies across customers
3. Transaction values are **independent** of frequency

In [ ]:
# Prepare features for Gamma-Gamma
def prepare_gg_features(df):
    """Prepare features for Gamma-Gamma model."""
    customer_values = df.groupby('Customer ID').agg({
        'Invoice': 'nunique',
        'Revenue': 'mean'
    }).reset_index()
    
    customer_values.columns = ['Customer ID', 'frequency', 'monetary_value']
    customer_values['frequency'] = customer_values['frequency'] - 1  # Exclude first
    
    return customer_values

gg_features = prepare_gg_features(train_df)

# Filter to repeat customers only (Gamma-Gamma requirement)
gg_features_repeat = gg_features[gg_features['frequency'] > 0]
print(f"Repeat customers for Gamma-Gamma: {len(gg_features_repeat):,}")

In [ ]:
# Fit Gamma-Gamma model
gg_model = GammaGammaFitter(penalizer_coef=0.01)
gg_model.fit(
    gg_features_repeat['frequency'],
    gg_features_repeat['monetary_value']
)

print("Gamma-Gamma Model Parameters:")
print(gg_model.params_)
print(f"\nModel Summary:\n{gg_model.summary}")

In [ ]:
# Predict average transaction value
gg_features['predicted_avg_value'] = gg_model.conditional_expected_average_profit(
    gg_features['frequency'].clip(lower=1),
    gg_features['monetary_value']
)

# Merge with BG/NBD features
clv_features = bgnbd_features.merge(gg_features[['Customer ID', 'predicted_avg_value']], 
                                     on='Customer ID', how='left')

# Calculate probabilistic CLV (12 months)
clv_features['clv_probabilistic'] = (
    clv_features['predicted_purchases_90d'] / 90 * 30 * 12 *  # Monthly purchases * 12 months
    clv_features['predicted_avg_value'] * 0.10  # Profit margin
)

print("CLV Features with Probabilistic Model:")
clv_features.head(10)

## 8. ML-Based CLV Prediction

Now let's build an ML model that combines all features for better prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb

# Prepare comprehensive features
def prepare_ml_features(df, snapshot_date=None):
    """Prepare comprehensive features for ML model."""
    if snapshot_date is None:
        snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
    
    features_list = []
    
    for customer_id, group in df.groupby('Customer ID'):
        invoices = group.groupby('Invoice').agg({
            'InvoiceDate': 'first',
            'Revenue': 'sum'
        }).reset_index()
        
        n_purchases = len(invoices)
        first_date = group['InvoiceDate'].min()
        last_date = group['InvoiceDate'].max()
        
        frequency = n_purchases - 1
        recency = (last_date - first_date).days
        T = (snapshot_date - first_date).days
        
        features_list.append({
            'Customer ID': customer_id,
            'frequency': frequency,
            'recency': recency,
            'T': T,
            'monetary_value': group['Revenue'].mean(),
            'total_revenue': group['Revenue'].sum(),
            'n_invoices': n_purchases,
            'n_unique_products': group['StockCode'].nunique(),
            'n_unique_countries': group['Country'].nunique(),
            'max_gap_days': invoices['InvoiceDate'].diff().dt.days.dropna().max() 
                           if len(invoices) > 1 else 0,
            'revenue_std': group['Revenue'].std() if len(group) > 1 else 0
        })
    
    return pd.DataFrame(features_list)

ml_features = prepare_ml_features(train_df)
print(f"ML features prepared for {len(ml_features):,} customers")

In [ ]:
# Add probabilistic model predictions as features
ml_features = ml_features.merge(
    clv_features[['Customer ID', 'predicted_purchases_90d', 'prob_alive', 'predicted_avg_value']],
    on='Customer ID',
    how='left'
)

# Add RFM scores
ml_features = ml_features.merge(
    rfm_segmented[['Customer ID', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']],
    on='Customer ID',
    how='left'
)

# Fill missing values
ml_features = ml_features.fillna(0)

print(f"Final feature set: {ml_features.shape}")
print(f"Features: {ml_features.columns.tolist()}")

In [ ]:
# Get holdout revenue (actual 90-day revenue)
holdout_revenue = test_df.groupby('Customer ID')['Revenue'].sum()
ml_features['holdout_revenue'] = ml_features['Customer ID'].map(holdout_revenue).fillna(0)

# Filter to customers with holdout revenue
train_mask = ml_features['holdout_revenue'] > 0
print(f"Customers with holdout revenue: {train_mask.sum()}")

# Prepare X and y
feature_cols = ['frequency', 'recency', 'T', 'monetary_value', 'total_revenue',
                'n_invoices', 'n_unique_products', 'max_gap_days', 'revenue_std',
                'predicted_purchases_90d', 'prob_alive', 'predicted_avg_value',
                'R_score', 'F_score', 'M_score']

X = ml_features.loc[train_mask, feature_cols]
y = ml_features.loc[train_mask, 'holdout_revenue']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Train XGBoost model
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# Predictions
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

# Metrics
def compute_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100 if np.any(y_true > 0) else np.nan
    
    print(f"\n{name}:")
    print(f"  MAE: £{mae:.2f}")
    print(f"  RMSE: £{rmse:.2f}")
    print(f"  R²: {r2:.4f}")
    print(f"  MAPE: {mape:.1f}%")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

train_metrics = compute_metrics(y_train, y_pred_train, "Training Set")
test_metrics = compute_metrics(y_test, y_pred_test, "Test Set")

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(
    importance_df,
    x='importance',
    y='feature',
    orientation='h',
    title='Feature Importance (XGBoost)',
    color='importance',
    color_continuous_scale='Blues'
)
fig.update_layout(showlegend=False, height=600)
fig.show()

In [ ]:
# Actual vs Predicted scatter plot
fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_test,
    mode='markers',
    marker=dict(opacity=0.6, color='steelblue'),
    name='Test Predictions'
))

# Perfect prediction line
max_val = max(y_test.max(), y_pred_test.max())
fig.add_trace(go.Scatter(
    x=[0, max_val],
    y=[0, max_val],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Perfect Prediction'
))

fig.update_layout(
    title='Actual vs Predicted 90-Day Revenue',
    xaxis_title='Actual Revenue (£)',
    yaxis_title='Predicted Revenue (£)',
    height=500
)

fig.show()

## 9. Model Comparison

Let's compare the probabilistic and ML approaches.

In [ ]:
# Add ML predictions to features
ml_features.loc[train_mask, 'ml_predicted_clv'] = xgb_model.predict(X)

# Compare models
test_customers = ml_features[train_mask].copy()

comparison = pd.DataFrame({
    'Actual': test_customers['holdout_revenue'],
    'Probabilistic (BG/NBD+GG)': test_customers['clv_probabilistic'],
    'ML (XGBoost)': test_customers['ml_predicted_clv']
})

# Calculate metrics for both
print("="*60)
print("MODEL COMPARISON")
print("="*60)

for model in ['Probabilistic (BG/NBD+GG)', 'ML (XGBoost)']:
    mae = mean_absolute_error(comparison['Actual'], comparison[model])
    rmse = np.sqrt(mean_squared_error(comparison['Actual'], comparison[model]))
    corr = np.corrcoef(comparison['Actual'], comparison[model])[0, 1]
    
    print(f"\n{model}:")
    print(f"  MAE: £{mae:.2f}")
    print(f"  RMSE: £{rmse:.2f}")
    print(f"  Correlation: {corr:.4f}")

In [ ]:
# Visualize comparison
fig = make_subplots(rows=1, cols=3, 
                    subplot_titles=['Distribution Comparison', 
                                   'Error Distribution', 
                                   'Model Correlation'])

# 1. Distribution comparison
fig.add_trace(go.Histogram(x=comparison['Actual'], name='Actual', 
                         marker_color='blue', opacity=0.5), row=1, col=1)
fig.add_trace(go.Histogram(x=comparison['Probabilistic (BG/NBD+GG)'], 
                         name='Probabilistic', marker_color='green', opacity=0.5), row=1, col=1)
fig.add_trace(go.Histogram(x=comparison['ML (XGBoost)'], 
                         name='ML', marker_color='orange', opacity=0.5), row=1, col=1)

# 2. Error distribution
prob_errors = comparison['Probabilistic (BG/NBD+GG)'] - comparison['Actual']
ml_errors = comparison['ML (XGBoost)'] - comparison['Actual']

fig.add_trace(go.Histogram(x=prob_errors, name='Prob Error', 
                         marker_color='green', opacity=0.7), row=1, col=2)
fig.add_trace(go.Histogram(x=ml_errors, name='ML Error', 
                         marker_color='orange', opacity=0.7), row=1, col=2)

# 3. Model correlation
fig.add_trace(go.Scatter(x=comparison['Probabilistic (BG/NBD+GG)'], 
                         y=comparison['ML (XGBoost)'],
                         mode='markers', marker=dict(opacity=0.5)), row=1, col=3)

fig.update_layout(height=400, showlegend=False)
fig.show()

## 10. Business Applications

### When to use each model:

| Model | Best For | Advantages | Disadvantages |
|-------|----------|------------|---------------|
| **BG/NBD + GG** | New customers, small data | Interpretable, probabilistic | Assumes stationarity |
| **ML (XGBoost)** | Large datasets, complex patterns | Higher accuracy | Needs more data, less interpretable |
| **Combined** | Production systems | Best of both worlds | More complex |

In [ ]:
# Marketing budget allocation by segment
budget_allocation = {
    'Champions': 0.30,
    'Loyal': 0.25,
    'Potential Loyalists': 0.20,
    'New Customers': 0.15,
    'At Risk': 0.25,
    'Hibernating': 0.10,
    'Lost': 0.05,
    'Needs Attention': 0.10
}

segment_actions = {
    'Champions': 'Loyalty rewards, VIP treatment, exclusive offers',
    'Loyal': 'Personalized recommendations, upselling, cross-selling',
    'Potential Loyalists': 'Targeted promotions, loyalty program enrollment',
    'New Customers': 'Welcome campaigns, onboarding, first-purchase incentives',
    'At Risk': 'Win-back campaigns, special discounts, personalized outreach',
    'Hibernating': 'Re-engagement offers, seasonal promotions, email campaigns',
    'Lost': 'Low-cost reactivation, exit surveys, generic offers',
    'Needs Attention': 'Segment-specific campaigns, A/B testing'
}

# Create summary table
summary = segment_stats.reset_index()
summary['Budget %'] = summary['segment'].map(budget_allocation)
summary['Recommended Action'] = summary['segment'].map(segment_actions)

print("MARKETING RECOMMENDATIONS BY SEGMENT")
print("="*100)

for _, row in summary.iterrows():
    print(f"\n📊 {row['segment']}")
    print(f"   Customers: {row['Count']:,} ({row['Revenue_Share_%']:.1f}% of revenue)")
    print(f"   Budget: {row['Budget %']*100:.0f}%")
    print(f"   Action: {row['Recommended Action']}")

In [ ]:
# Calculate CLV-based customer acquisition budget
avg_clv = ml_features['clv_probabilistic'].mean()
avg_ml_clv = ml_features['ml_predicted_clv'].mean()
total_customers = len(ml_features)

print("\nCUSTOMER VALUE SUMMARY")
print("="*60)
print(f"Average Probabilistic CLV: £{avg_clv:.2f}")
print(f"Average ML CLV: £{avg_ml_clv:.2f}")
print(f"Total Customers: {total_customers:,}")
print(f"Total CLV Potential: £{avg_clv * total_customers:,.2f}")

# Customer acquisition ROI
cac_typical = 50  # Assume £50 CAC
clv_cac_ratio = avg_clv / cac_typical

print(f"\nCUSTOMER ACQUISITION ROI")
print("="*60)
print(f"Typical CAC: £{cac_typical}")
print(f"CLV:CAC Ratio: {clv_cac_ratio:.1f}:1")
print(f"Margin: £{avg_clv - cac_typical:.2f} per customer")

if clv_cac_ratio >= 3:
    print("✅ CLV:CAC ratio is healthy (≥3:1)")
elif clv_cac_ratio >= 1:
    print("⚠️ CLV:CAC ratio is acceptable but could be improved")
else:
    print("❌ CLV:CAC ratio is below break-even - reduce CAC or increase CLV")

In [ ]:
# At-risk high-value customers
at_risk = ml_features[
    (ml_features['prob_alive'] < 0.5) &
    (ml_features['clv_probabilistic'] > avg_clv)
].sort_values('clv_probabilistic', ascending=False)

print(f"\nAT-RISK HIGH-VALUE CUSTOMERS")
print("="*60)
print(f"Found {len(at_risk):,} customers at risk with high CLV potential")
print(f"\nTop 5 at-risk customers:")
print(at_risk[['Customer ID', 'clv_probabilistic', 'prob_alive', 'segment']].head())

## 11. Summary and Next Steps

### Key Takeaways:

1. **RFM Segmentation** provides quick customer insights
2. **BG/NBD + Gamma-Gamma** models offer interpretable probabilistic predictions
3. **ML models** (XGBoost) provide higher accuracy with proper feature engineering
4. **Combined approach** is best for production systems
5. **CLV insights** enable smarter marketing budget allocation

### Next Steps:

1. **Deploy to Production** - Use the FastAPI and Streamlit apps
2. **Set Up Monitoring** - Track prediction accuracy over time
3. **A/B Test Campaigns** - Validate segment-specific strategies
4. **Retrain Periodically** - Update models with new data
5. **Explore Deep Learning** - Consider neural networks for complex patterns

In [ ]:
# Save models (optional - requires joblib)
import joblib

# Save for later use
# joblib.dump(bgnbd_model, 'models/bgnbd_model.pkl')
# joblib.dump(gg_model, 'models/gamma_gamma_model.pkl')
# joblib.dump(xgb_model, 'models/xgb_model.pkl')

print("Models can be saved for production deployment.")
print("\nTo deploy:")
print("1. Run: python -m src.train")
print("2. Start API: uvicorn api.main:app --reload")
print("3. Start Dashboard: streamlit run app.py")